# 🍌 Banana Detection — YOLOv8 Training

**Dataset:** 1005 gambar pisang (mentah, mengkal, matang, busuk)  
**Model:** YOLOv8n (nano) — ringan, cocok untuk Android  

---

### Sebelum mulai:
1. Pastikan Runtime → Change runtime type → **GPU (T4)**
2. Upload dataset ke Google Drive terlebih dahulu
3. Struktur folder di Drive harus seperti ini:

```
MyDrive/
  banana_dataset/
    train/
      images/   <- semua gambar train
      labels/   <- semua label train (.txt)
    val/
      images/   <- semua gambar val
      labels/   <- semua label val (.txt)
    dataset.yaml
```

**Jalankan script `fix_dataset_names.py` di komputer lokal dulu** untuk merapikan dataset sebelum upload!

## ⚙️ 1. Konfigurasi

In [ ]:
# ============================================================
# SESUAIKAN KONFIGURASI DI SINI
# ============================================================

# Path folder dataset di Google Drive (setelah di-mount)
DATASET_PATH = '/content/drive/MyDrive/banana_dataset'

# Model YOLOv8 yang dipakai
# Pilihan: yolov8n.pt (paling ringan), yolov8s.pt, yolov8m.pt, yolov8l.pt
BASE_MODEL = 'yolov8n.pt'

# Nama project & experiment (hasil training tersimpan di sini)
PROJECT_NAME = 'banana_detection'
RUN_NAME     = 'v2_yolov8n'

# Hyperparameter training
EPOCHS      = 100
IMAGE_SIZE  = 640
BATCH_SIZE  = 16    # turunkan ke 8 jika GPU out of memory
PATIENCE    = 20    # early stopping: stop jika tidak ada improvement selama N epoch

# ============================================================
print('✅ Konfigurasi selesai')
print(f'   Dataset  : {DATASET_PATH}')
print(f'   Model    : {BASE_MODEL}')
print(f'   Epochs   : {EPOCHS}')
print(f'   Img size : {IMAGE_SIZE}')
print(f'   Batch    : {BATCH_SIZE}')

## 📂 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive berhasil di-mount')

## 🔍 3. Cek Dataset

In [ ]:
import os

yaml_path = os.path.join(DATASET_PATH, 'dataset.yaml')

# Cek struktur folder
for split in ['train', 'val']:
    img_dir = os.path.join(DATASET_PATH, split, 'images')
    lbl_dir = os.path.join(DATASET_PATH, split, 'labels')
    n_img   = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]) if os.path.isdir(img_dir) else 0
    n_lbl   = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')]) if os.path.isdir(lbl_dir) else 0
    print(f'{split:5s} -> {n_img:4d} images | {n_lbl:4d} labels', '✅' if n_img == n_lbl else '⚠️ JUMLAH TIDAK SAMA!')

print()

# Tampilkan dataset.yaml
print('=== dataset.yaml ===')
with open(yaml_path) as f:
    print(f.read())

# Cek distribusi kelas
print('=== Distribusi kelas ===')
class_names = {0:'mentah', 1:'mengkal', 2:'matang', 3:'busuk'}
counts = {k:0 for k in class_names}
for split in ['train', 'val']:
    lbl_dir = os.path.join(DATASET_PATH, split, 'labels')
    if not os.path.isdir(lbl_dir):
        continue
    for fname in os.listdir(lbl_dir):
        if not fname.endswith('.txt'):
            continue
        with open(os.path.join(lbl_dir, fname)) as f:
            for line in f:
                cls = int(line.split()[0])
                if cls in counts:
                    counts[cls] += 1
for cls_id, name in class_names.items():
    print(f'  Class {cls_id} ({name:8s}): {counts[cls_id]:4d} bbox')

## 📦 4. Install Ultralytics

In [ ]:
!pip install ultralytics -q
import ultralytics
ultralytics.checks()
print('✅ Ultralytics siap')

## 🚀 5. Training

In [ ]:
import os
from ultralytics import YOLO

yaml_path = os.path.join(DATASET_PATH, 'dataset.yaml')

# Load pretrained model
model = YOLO(BASE_MODEL)

# Mulai training!
results = model.train(
    data       = yaml_path,
    epochs     = EPOCHS,
    imgsz      = IMAGE_SIZE,
    batch      = BATCH_SIZE,
    patience   = PATIENCE,
    project    = PROJECT_NAME,
    name       = RUN_NAME,
    device     = 0,            # 0 = GPU
    workers    = 2,
    cache      = True,
    optimizer  = 'AdamW',
    lr0        = 0.001,
    lrf        = 0.01,
    weight_decay = 0.0005,
    augment    = True,
    fliplr     = 0.5,
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    degrees    = 10,
    translate  = 0.1,
    scale      = 0.5,
    mosaic     = 1.0,
    save_period = 10,          # simpan checkpoint setiap N epoch
    verbose    = True,
)

print('\n✅ Training selesai!')
print(f'   mAP50   : {results.results_dict.get("metrics/mAP50(B)", 0):.4f}')
print(f'   mAP50-95: {results.results_dict.get("metrics/mAP50-95(B)", 0):.4f}')

## 📊 6. Evaluasi & Lihat Hasil

In [ ]:
import glob
from IPython.display import Image, display

run_dir = os.path.join(PROJECT_NAME, RUN_NAME)
print(f'Hasil training ada di: {run_dir}')

# Tampilkan confusion matrix
for img_path in ['confusion_matrix.png', 'confusion_matrix_normalized.png', 'results.png', 'PR_curve.png', 'F1_curve.png']:
    full = os.path.join(run_dir, img_path)
    if os.path.exists(full):
        print(f'--- {img_path} ---')
        display(Image(full, width=800))

## 🧪 7. Validasi Model

In [ ]:
best_weights = os.path.join(PROJECT_NAME, RUN_NAME, 'weights', 'best.pt')
model_best = YOLO(best_weights)

val_results = model_best.val(
    data    = yaml_path,
    imgsz   = IMAGE_SIZE,
    batch   = BATCH_SIZE,
    device  = 0,
    verbose = True,
)

print('\n=== Hasil Validasi Per Kelas ===')
class_names_list = ['mentah', 'mengkal', 'matang', 'busuk']
for i, name in enumerate(class_names_list):
    try:
        ap50 = float(val_results.box.ap50[i])
        print(f'  {name:8s}: AP@50 = {ap50:.4f}')
    except:
        pass

## 💾 8. Export Model & Simpan ke Drive

In [ ]:
import shutil

best_weights = os.path.join(PROJECT_NAME, RUN_NAME, 'weights', 'best.pt')

# Simpan best.pt ke Google Drive
save_dir = '/content/drive/MyDrive/banana_model'
os.makedirs(save_dir, exist_ok=True)

dest_pt = os.path.join(save_dir, 'banana_detection_v2.pt')
shutil.copy2(best_weights, dest_pt)
print(f'✅ Model PyTorch disimpan: {dest_pt}')

# Export ke format TFLite (untuk Android)
print('\nMengekspor ke TFLite untuk Android...')
model_best = YOLO(best_weights)
model_best.export(
    format  = 'tflite',
    imgsz   = IMAGE_SIZE,
    int8    = False,
    half    = False,
)

# Cari file tflite yang dihasilkan
tflite_files = glob.glob(os.path.join(PROJECT_NAME, RUN_NAME, 'weights', '*.tflite'))
if tflite_files:
    dest_tflite = os.path.join(save_dir, 'banana_detection_v2.tflite')
    shutil.copy2(tflite_files[0], dest_tflite)
    print(f'✅ Model TFLite disimpan: {dest_tflite}')
else:
    print('⚠️ File TFLite tidak ditemukan, coba export manual.')

# Tampilkan ukuran file
for f in [dest_pt] + ([dest_tflite] if tflite_files else []):
    size_mb = os.path.getsize(f) / (1024*1024)
    print(f'   {os.path.basename(f)}: {size_mb:.1f} MB')

## ✅ Selesai!

File yang sudah tersimpan di Google Drive (`banana_model/`):
- **`banana_detection_v2.pt`** — Model PyTorch (untuk backend Flask)
- **`banana_detection_v2.tflite`** — Model TFLite (untuk Android)

### Langkah selanjutnya:
1. Download `banana_detection_v2.pt` dari Drive
2. Ganti file lama di `trained_models/banana_detection_v1/weights/best.pt`
3. Restart backend Flask — model baru langsung aktif!